# Pipeline Đánh giá CoT LLM
## Giải toán tiếng Việt với Llama-3.2-1B

Notebook này triển khai một pipeline đánh giá hoàn chỉnh để so sánh phản hồi từ mô hình Llama ở chế độ Trực tiếp (non-CoT) và Chuỗi Suy luận (CoT) trên các bài toán tiếng Việt.

## 1. Thiết lập & Import

In [ ]:
import gc
import os
import re
from pathlib import Path
from textwrap import dedent

import torch
from datasets import Dataset
from dotenv import load_dotenv
from huggingface_hub import login
from openpyxl import load_workbook
from transformers import logging as hf_logging, pipeline

# Ẩn các cảnh báo từ HuggingFace
hf_logging.set_verbosity_error()

# Tải các biến môi trường
load_dotenv()

## 2. Cấu hình & Hằng số

In [ ]:
# --- Cấu hình Dataset & Model ---
DATASET_FILE_PATH = "data/sample_dataset.xlsx"
DATASET_NAME = DATASET_FILE_PATH  # Bí danh tương thích ngược
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
BATCH_SIZE = 5
PIPE_BATCH_SIZE = 5

# --- Tham số Inference ---
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.1  # Giá trị thấp để tăng độ chính xác toán học
TOP_P = 0.95

# --- Các dấu Prompt ---
SOLUTION_START = "####"
SOLUTION_END = ""
REASONING_START = "<thought>"
REASONING_END = "</thought>"

print(f"Configuration:")
print(f"  Dataset: {DATASET_FILE_PATH}")
print(f"  Model: {MODEL_ID}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Device: {'GPU (bfloat16)' if torch.cuda.is_available() else 'CPU (float32)'}")

## 3. Định nghĩa Prompt

In [ ]:
# Prompt trực tiếp (không Chain-of-Thought)
DIRECT_PROMPT = f"""
    Bạn là một trợ lý giải toán cực kỳ chính xác.
    Bạn sẽ nhận được một bài toán bằng tiếng Việt.

    Hãy đọc kỹ đề bài và CHỈ trả về đáp án số cuối cùng.
    Đặt đáp án ngay sau {SOLUTION_START}.
    Không giải thích, không trình bày bước làm, không thêm bất kỳ nội dung nào khác.
"""

# Prompt Chain-of-Thought
COT_PROMPT = f"""
    Bạn là một trợ lý giải toán cực kỳ chính xác.
    Bạn sẽ nhận được một bài toán bằng tiếng Việt.

    Hãy làm đúng theo các bước sau:
    1. Suy luận từng bước bằng tiếng Việt.
    2. Viết phần suy luận nằm giữa {REASONING_START} và {REASONING_END}.
    3. Sau đó chỉ đưa ra đáp án số cuối cùng, đặt sau {SOLUTION_START}.

    Không được xuất thêm nội dung ngoài phần suy luận và đáp án.
"""

print("Prompts defined:")
print(f"  - DIRECT_PROMPT: {len(DIRECT_PROMPT)} characters")
print(f"  - COT_PROMPT: {len(COT_PROMPT)} characters")

## 4. Các hàm tiện ích

In [ ]:
def extract_answer(text: str) -> str:
    """Trích xuất số đáp án từ văn bản phản hồi.
    
    Ưu tiên mẫu `####{answer}` cuối cùng, rồi dùng các hình thức số khác.
    """
    # Mẫu: #### {số} ở cuối văn bản (ưu tiên cao nhất)
    m = re.search(r"####\s*([+-]?[\d,]+(?:\.\d+)?\s*)$", text)
    if m:
        return m.group(1).strip().replace(",", "")

    # Mẫu: #### {số} ở bất kỳ đâu trong văn bản
    m = re.search(r"####\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Mẫu: "Đáp án là: {số}"
    m = re.search(r"Đáp án là:\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Mẫu: "Đáp án là: {số}" (tiếng Anh)
    m = re.search(r"[Tt]he answer is\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Dự phòng: trích xuất số cuối cùng trong văn bản
    nums = re.findall(r"[+-]?[\d,]+(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else ""


def decode_pipe_output(outputs):
    """Giải mã đầu ra từ HuggingFace pipeline thành danh sách chuỗi văn bản."""
    texts = []
    for out in outputs:
        if isinstance(out, dict) and "generated_text" in out:
            texts.append(out["generated_text"])
        elif isinstance(out, list) and len(out) > 0 and "generated_text" in out[0]:
            texts.append(out[0]["generated_text"])
        elif isinstance(out, str):
            texts.append(out)
        else:
            texts.append(str(out))
    return texts

## 5. Các hàm tải tập dữ liệu

In [ ]:
def _resolve_dataset_path(dataset_path: str) -> Path:
    """Phân giải đường dẫn dataset tương đối với vị trí notebook."""
    path = Path(dataset_path)
    if path.is_absolute():
        return path
    # Trong ngữ cảnh notebook, sử dụng thư mục làm việc hiện tại
    return Path.cwd() / path


def load_dataset_from_excel(dataset_path: str):
    """Tải tập dữ liệu Excel cục bộ và chuẩn hóa các cột."""
    path = _resolve_dataset_path(dataset_path)
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.active

    headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
    if "query_vi" not in headers:
        raise ValueError(f"Thiếu cột bắt buộc 'query_vi' trong {path}")

    response_header = next(
        (h for h in headers if isinstance(h, str) and h.startswith("response_vi")),
        None,
    )
    if response_header is None:
        raise ValueError(f"Thiếu cột phản hồi bắt buộc trong {path}")

    rows = []
    for values in ws.iter_rows(min_row=2, values_only=True):
        row = {}
        for header, value in zip(headers, values):
            if header == "query_vi":
                row["query_vi"] = value
            elif header == response_header:
                row["response_vi"] = value
        if row:
            rows.append(row)

    wb.close()
    return Dataset.from_list(rows)


def add_ground_truth(ds):
    """Trích xuất các câu trả lời đúng từ cột phản hồi."""
    return ds.map(
        lambda example: {"ground_truth": extract_answer(example["response_vi"])},
        remove_columns=[],
    )

## 6. Xây dựng Pipeline & Sinh text

In [ ]:
def get_hf_token():
    """Lấy mã token HuggingFace từ môi trường."""
    return os.getenv("HF_TOKEN")


def build_pipeline():
    """Xây dựng và trả về HuggingFace pipeline với thiết bị và xác thực phù hợp."""
    hf_token = get_hf_token()
    if hf_token:
        login(token=hf_token)
    else:
        print("Lưu ý: Không tìm thấy HF_TOKEN trong môi trường. Vui lòng đăng nhập thủ công.")

    device = 0 if torch.cuda.is_available() else -1
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    pipe = pipeline(
        "text-generation",
        model=MODEL_ID,
        device=device,
        dtype=dtype,
    )

    # Các tokenizer kiểu Llama thường không xác định pad token theo mặc định.
    # Tạo batch yêu cầu pad token, vì vậy tái sử dụng EOS và left-pad đầu vào.
    if pipe.tokenizer.pad_token_id is None:
        pipe.tokenizer.pad_token = pipe.tokenizer.eos_token
        pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id
    pipe.tokenizer.padding_side = "left"

    return pipe


def generate_text_with_prompt(pipe, prompts, system_prompt: str, batch_size: int = PIPE_BATCH_SIZE):
    """Sinh văn bản bằng cách sử dụng chat template với system prompt.

    Tham số:
        pipe: HuggingFace pipeline
        prompts: Danh sách các prompt từ người dùng
        system_prompt: Thông điệp hệ thống cho mô hình
        batch_size: Kích thước batch cho pipeline

    Trả về:
        Danh sách các chuỗi văn bản được sinh ra
    """
    formatted_prompts = []
    for user_prompt in prompts:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_str = pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        formatted_prompts.append(prompt_str)

    outputs = pipe(
        formatted_prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        return_full_text=False,
        batch_size=batch_size,
    )

    return decode_pipe_output(outputs)

## 7. Pipeline đánh giá chính

In [ ]:
def run():
    print("=== Starting Evaluation Pipeline ===\n")

    # --- Tải và chuẩn bị dataset ---
    print(f"1. Loading dataset from {DATASET_FILE_PATH}...")
    ds = load_dataset_from_excel(DATASET_FILE_PATH)
    print(f"   Dataset loaded with {len(ds)} samples")
    print("   Using the prepared 15-row dataset directly; no further sampling.\n")

    # Thêm ground truth được trích xuất từ response_vi
    ds = add_ground_truth(ds)

    # --- Xây dựng pipeline ---
    print("\n2. Building HuggingFace pipeline...")
    pipe = build_pipeline()
    print(f"   Model: {MODEL_ID}")
    print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU'}\n")

    # --- Chạy đánh giá ---
    print("3. Running batch evaluation...")
    print(f"   Total samples: {len(ds)}, Batch size: {BATCH_SIZE}\n")

    # Xử lý các batch
    for batch_idx, start in enumerate(range(0, len(ds), BATCH_SIZE), start=1):
        end = min(start + BATCH_SIZE, len(ds))
        batch = ds.select(range(start, end))

        queries = list(batch["query_vi"])
        ground_truths = list(batch["ground_truth"])

        # Sinh phản hồi: không-CoT và với-CoT
        print(f"   Batch {batch_idx}: Generating non-CoT responses...")
        no_cot_texts = generate_text_with_prompt(
            pipe, queries, DIRECT_PROMPT, batch_size=BATCH_SIZE
        )

        print(f"   Batch {batch_idx}: Generating CoT responses...")
        cot_texts = generate_text_with_prompt(
            pipe, queries, COT_PROMPT, batch_size=BATCH_SIZE
        )

        for idx, (q, gt, no_text, cot_text) in enumerate(
            zip(queries, ground_truths, no_cot_texts, cot_texts),
            start=start + 1,
        ):
            ans_no = extract_answer(no_text)
            ans_cot = extract_answer(cot_text)

            print(f"\n   Sample {idx}")
            print(f"     query_vi: {q}")
            print(f"     ground_truth: {gt}")
            print(f"     non-CoT answer: {ans_no}")
            print(f"     CoT answer: {ans_cot}")

        # Giải phóng bộ nhớ
        del no_cot_texts, cot_texts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- Tóm tắt ---
    print("\n4. Final Summary")
    print(f"   Total samples processed: {len(ds)}")
    print("\n=== Evaluation Complete ===")

## 8. Chạy đánh giá

In [ ]:
# Chạy pipeline đánh giá
run()